# Fully Connected Networks with PyTorch

A practical CIFAR-10 multilayer perceptron (MLP) notebook focused on useful PyTorch patterns: `nn.Module`, `nn.Linear`, initialization, batch normalization, dropout, autograd, `DataLoader`, training/evaluation modes, optimizers, and `state_dict` checkpoints. PyTorch implements standard layers, losses, and gradients; the notebook avoids hand-written layers, optimizers, and solver frameworks.

Run from this notebook's `A3` directory so the course `eecs598` data helper is importable. The model and training loop are standard PyTorch and can be adapted to other `Dataset`s and `DataLoader`s.

An MLP flattens each image and therefore does not exploit the spatial structure that a CNN can use; this is a useful baseline, not the strongest CIFAR-10 architecture.


## Setup and reproducibility

In [ ]:
import copy
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

import eecs598


def set_seed(seed: int = 0) -> None:
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Load CIFAR-10

In [ ]:
# Keep tensors on CPU; batches are transferred to the accelerator in the loop.
data = eecs598.data.preprocess_cifar10(
    cuda=False, dtype=torch.float32, flatten=False
)
for split, images, labels in (
    ('train', data['X_train'], data['y_train']),
    ('validation', data['X_val'], data['y_val']),
    ('test', data['X_test'], data['y_test']),
):
    print(f'{split:10s} images={tuple(images.shape)}, labels={tuple(labels.shape)}')


## Define an MLP with `nn.Module`

`nn.Linear` consumes the last input dimension. `nn.Flatten` converts each NCHW image into a feature vector. Batch normalization stabilizes hidden activations, while dropout regularizes hidden layers during training and disables itself in evaluation mode.


In [ ]:
class FullyConnectedNet(nn.Module):
    def __init__(
        self,
        input_dim: int = 3 * 32 * 32,
        hidden_dims: tuple[int, ...] = (512, 256),
        num_classes: int = 10,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        if not 0.0 <= dropout < 1.0:
            raise ValueError('dropout must be in [0, 1)')

        layers: list[nn.Module] = [nn.Flatten()]
        in_features = input_dim
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(in_features, hidden_dim, bias=False),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(inplace=True),
            ])
            if dropout > 0:
                layers.append(nn.Dropout(p=dropout))
            in_features = hidden_dim
        layers.append(nn.Linear(in_features, num_classes))
        self.network = nn.Sequential(*layers)
        self.apply(self._initialize)

    @staticmethod
    def _initialize(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.kaiming_normal_(module.weight, nonlinearity='relu')
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.network(x)


model = FullyConnectedNet().to(device)
print(model)


## Autograd and gradient checking

For standard layers, `loss.backward()` is the implementation. `gradcheck` is useful when authoring a custom differentiable operation: it compares autograd's gradients with finite differences on small double-precision inputs.


In [ ]:
import torch.nn.functional as F

x = torch.randn(2, 4, dtype=torch.double, requires_grad=True)
weight = torch.randn(3, 4, dtype=torch.double, requires_grad=True)
bias = torch.randn(3, dtype=torch.double, requires_grad=True)

assert torch.autograd.gradcheck(F.linear, (x, weight, bias))
print('Linear-layer gradient check passed')


## DataLoaders and optimization

In [ ]:
batch_size = 128
train_loader = DataLoader(
    TensorDataset(data['X_train'], data['y_train']),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
val_loader = DataLoader(
    TensorDataset(data['X_val'], data['y_val']),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)
test_loader = DataLoader(
    TensorDataset(data['X_test'], data['y_test']),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == 'cuda'),
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
epochs = 10
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


## A conventional training and evaluation loop

The loop shows the standard PyTorch steps explicitly. Gradients accumulate by default, so clear them before each update. Use `model.train()` during optimization and `model.eval()` plus `inference_mode()` during evaluation; this changes dropout and batch-normalization behavior.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    correct = 0
    count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += batch_size

    return total_loss / count, correct / count


@torch.inference_mode()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    count = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        total_loss += criterion(logits, labels).item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        count += labels.size(0)

    return total_loss / count, correct / count


## Train, select by validation performance, and save

In [ ]:
history = {'train_loss': [], 'train_accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_accuracy = 0.0
best_state = copy.deepcopy(model.state_dict())

for epoch in range(epochs):
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_accuracy)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = copy.deepcopy(model.state_dict())

    print(
        f'Epoch {epoch + 1:02d}/{epochs} | '
        f'train loss {train_loss:.4f}, acc {train_accuracy:.3f} | '
        f'val loss {val_loss:.4f}, acc {val_accuracy:.3f}'
    )

model.load_state_dict(best_state)
checkpoint_path = Path('best_cifar10_mlp.pt')
torch.save(model.state_dict(), checkpoint_path)
print(f'Saved best validation model to {checkpoint_path.resolve()}')


## Inspect learning curves and test once

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='validation')
axes[0].set(title='Cross-entropy loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[1].plot(history['train_accuracy'], label='train')
axes[1].plot(history['val_accuracy'], label='validation')
axes[1].set(title='Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend()
plt.tight_layout()

# Keep the test split out of model selection; evaluate it after choosing the checkpoint.
test_loss, test_accuracy = evaluate(model, test_loader, criterion, device)
print(f'Test loss: {test_loss:.4f} | test accuracy: {test_accuracy:.3f}')


## Reload a `state_dict`

Save model weights rather than serializing the Python model object. Recreate the architecture first, then load the weights; checkpoints remain portable as long as the model definition is available.


In [ ]:
reloaded_model = FullyConnectedNet().to(device)
state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
reloaded_model.load_state_dict(state_dict)
reloaded_model.eval()
print('Checkpoint loaded into a fresh model')


## What is worth remembering

- Use `nn.Module` and `nn.Sequential` to compose registered layers; flatten image batches before fully connected layers.
- `nn.Linear` stores weights as `(out_features, in_features)`; PyTorch handles the matrix operation and gradients.
- Use suitable initialization for the activation (`kaiming_normal_` for ReLU networks); batch-normalization parameters have sensible defaults.
- Apply `nn.Dropout` for regularization and switch to `eval()` for inference.
- The update step is `zero_grad` ? forward ? loss ? `backward` ? optimizer `step`.
- Use validation data for model selection and reserve the test split for final evaluation.
- Save/load `state_dict`s. Use `gradcheck` for custom autograd code, not to reimplement standard layers or optimizers.
